<h1 style="
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    font-size: 36px;
    color: #ffffff;
    background-color: #9E1B32;
    padding: 20px;
    border-radius: 12px;
    text-align: center;
    box-shadow: 0px 4px 10px rgba(0, 0, 0, 0.1);">
    Forcing Evaluation
</h1>


**Authors:** 

<ul style="line-height:1.5;">
<li>Parthkumar Modi <a href="mailto:pamodi@ua.edu">(pamodi@ua.edu)</a></li>
</ul>

**Last Updated:** 10/10/2026

**Purpose:**

This notebook evaluates how well two gridded rainfall products, **AORC** and **IMERG**, match rain gauge observations, hour by hour. These products drive the NextGen model, so errors in rainfall carry into the simulated streamflow.

**Audience:**

Researchers and practitioners with some experience in Jupyter Notebooks, basic Python, and basic hydrologic data analysis.

**Description:**

We map the basin and its hourly rain gauges, remove bad gauge data with four simple quality checks, and score each product against each gauge (bias ratio, correlation, KGE, POD, FAR, CSI and heavy-rain ratio). We then compare them visually: cumulative rainfall, hourly scatter, KGE at each gauge, one storm hour by hour, and KGE year by year. A test is marked **not independent** when the product may already contain that gauge.

**Data Description:**


We check two gridded rainfall products, **hour by hour**, against the rain gauges in and around one small basin,
**Sipsey Fork near Grayson, Alabama** (USGS stream gauge **02450250**, 243 km²), over **2002–2022**.

<table style="border-collapse: collapse; font-size: 14px;">
  <thead>
    <tr style="background-color: #ffffff; color: #9E1B32;">
      <th style="padding: 8px; text-align: center; vertical-align: middle; color: #9E1B32; border-bottom: 2px solid #9E1B32;">Product</th>
      <th style="padding: 8px; text-align: center; vertical-align: middle; color: #9E1B32; border-bottom: 2px solid #9E1B32;">What it is</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="padding: 8px; text-align: center; vertical-align: middle;"><strong>IMERG</strong></td><td style="padding: 8px; text-align: center; vertical-align: middle;">NASA's satellite rainfall (GPM IMERG Final Run, version 07): many satellites merged, then adjusted each month to rain-gauge totals. ~10 km grid.</td></tr>
    <tr><td style="padding: 8px; text-align: center; vertical-align: middle;"><strong>AORC</strong></td><td style="padding: 8px; text-align: center; vertical-align: middle;">NOAA's Analysis of Record for Calibration: a long, consistent hourly rainfall record built for hydrologic models. ~1 km grid.</td></tr>
  </tbody>
</table>

**The data are premade** (folder `data/`), so nothing is downloaded today: one table with every hour of
2002–2022 for every hourly gauge, and both products at each gauge's grid cell.

<div style="background:#9E1B32; border-left:6px solid #F0627A; padding:12px 16px; border-radius:8px; margin-top:0; margin-bottom:0;">
  <h3 style="margin:0; font-size:20px; font-weight:700; color:#fff5f7;">
    1. Check the Python packages and set the variables
  </h3>
</div>

If a package is missing, the cell tells you what to install. Install it, then run the cell again.

In [ ]:
# Step 0: check that every package this notebook needs is installed.
from utils.forcing_evaluation_utils import check_packages

# Step 1: settings and imports.
import warnings
from pathlib import Path

import pandas as pd

from utils.forcing_evaluation_utils import (load_data, plot_map, quality_control, hourly_scores, plot_cumulative_rainfall,
                                      plot_hourly_scatter, plot_kge_by_gauge, plot_wettest_storm, plot_kge_per_year)

warnings.filterwarnings("ignore", category=RuntimeWarning)   # keep the output tidy
pd.set_option("display.width", 200)

DATA = Path("data")                   # the ready-made data (made by prepare_workshop_data.py)

# ---------------------------------------------------------------- the period to look at (anywhere in 2002-2022)
START = "2002-01-01"
END = "2022-12-31"

# ---------------------------------------------------------------- products
PRODUCTS = ["imerg", "aorc"]
LABEL = {"imerg": "IMERG", "aorc": "AORC"}
COLOUR = {"gauge": "black", "imerg": "#1b4f80", "aorc": "#c0622b"}

# Is a gauge-vs-product test "independent"? Not if the product may already contain that gauge.
#   IMERG     is adjusted only once a month, to a global gauge analysis (GPCC) built mostly from airport-type
#             stations: it never sees the USGS or HADS gauges, and airports only as monthly totals.
#   AORC      is built partly from gauge-based analyses, so it may contain them too.
NETWORKS_USED_BY = {
    "imerg": ["ASOS"],
    "aorc": ["USGS", "HADS", "ASOS"],
}

# ---------------------------------------------------------------- quality control of the gauges (Step 4)
MAX_RAIN_IN_ONE_HOUR_MM = 100   # 1. more rain than this in one hour is impossible here: treated as an error
BLOCKED_BELOW_PERCENT = 10      # 2. a month when the gauge caught less than 10% of the rain IMERG saw: blocked
CLOCK_CHECK_HOURS = 3           # 3. compare the gauge with IMERG shifted up to 3 hours either way: best must be 0
MIN_YEARS_OF_DATA = 2           # 4. a gauge needs at least 2 years of data (after checks 1-2) to be used

# ---------------------------------------------------------------- scores (Step 5)
WET_HOUR_MM = 1.0            # an hour with at least this much is a "rain" hour (for hits, misses, false alarms)
HEAVY_HOUR_MM = 10.0         # a "heavy" hour
print("Settings ready.")

check_packages()

<div style="background:#9E1B32; border-left:6px solid #F0627A; padding:12px 16px; border-radius:8px; margin-top:0; margin-bottom:0;">
  <h3 style="margin:0; font-size:20px; font-weight:700; color:#fff5f7;">
    2. Load the data
  </h3>
</div>

<table style="border-collapse: collapse; font-size: 14px;">
  <thead>
    <tr style="background-color: #ffffff; color: #9E1B32;">
      <th style="padding: 8px; text-align: center; vertical-align: middle; color: #9E1B32; border-bottom: 2px solid #9E1B32;">File</th>
      <th style="padding: 8px; text-align: center; vertical-align: middle; color: #9E1B32; border-bottom: 2px solid #9E1B32;">What is in it</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="padding: 8px; text-align: center; vertical-align: middle;"><code>sipsey_fork_gauges.csv</code></td><td style="padding: 8px; text-align: center; vertical-align: middle;">every rain gauge in and around the basin: network, ID, location, distance outside the basin</td></tr>
    <tr><td style="padding: 8px; text-align: center; vertical-align: middle;"><code>sipsey_fork_hourly.csv</code></td><td style="padding: 8px; text-align: center; vertical-align: middle;">one row per hour: each hourly gauge (<code>gauge_&lt;id&gt;</code>), and IMERG and AORC at that gauge's grid cell (<code>imerg_&lt;id&gt;</code>, <code>aorc_&lt;id&gt;</code>)</td></tr>
  </tbody>
</table>

<div style="background-color:rgba(184,34,61,0.12); border-left: 5px solid #E04A63; padding: 14px; border-radius: 8px; font-family: 'Segoe UI', sans-serif; font-size: 15px; margin-top: 10px;">
  <strong>Times:</strong> an hourly value is the rain in the hour <strong>ending</strong> at its time, in UTC (14:00 = 13:00–14:00).
</div>

In [ ]:
# Step 2: load the ready-made data (hourly gauges only).
info, gauges, hourly = load_data(DATA, START, END)
BASIN, BASIN_NAME = info["basin"], info["basin_name"]
hourly.head()

<div style="background:#9E1B32; border-left:6px solid #F0627A; padding:12px 16px; border-radius:8px; margin-top:0; margin-bottom:0;">
  <h3 style="margin:0; font-size:20px; font-weight:700; color:#fff5f7;">
    3. Map: the basin, its streams and the hourly gauges
  </h3>
</div>

We use every public hourly rain gauge inside the basin or within a small buffer around it (dashed line). The buffer
grows with the basin: half the radius of a circle with the basin's area (4.4 km here).

In [ ]:
# Step 3: map.
plot_map(DATA, info, gauges, BASIN, BASIN_NAME)

<div style="background:#9E1B32; border-left:6px solid #F0627A; padding:12px 16px; border-radius:8px; margin-top:0; margin-bottom:0;">
  <h3 style="margin:0; font-size:20px; font-weight:700; color:#fff5f7;">
    4. Quality control of the gauges
  </h3>
</div>

Gauges are not perfect. Four simple checks, one setting each:

<table style="border-collapse: collapse; font-size: 14px;">
  <thead>
    <tr style="background-color: #ffffff; color: #9E1B32;">
      <th style="padding: 8px; text-align: center; vertical-align: middle; color: #9E1B32; border-bottom: 2px solid #9E1B32;">Check</th>
      <th style="padding: 8px; text-align: center; vertical-align: middle; color: #9E1B32; border-bottom: 2px solid #9E1B32;">What it does</th>
      <th style="padding: 8px; text-align: center; vertical-align: middle; color: #9E1B32; border-bottom: 2px solid #9E1B32;">Why</th>
      <th style="padding: 8px; text-align: center; vertical-align: middle; color: #9E1B32; border-bottom: 2px solid #9E1B32;">Setting</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="padding: 8px; text-align: center; vertical-align: middle;">1. Impossible values</td><td style="padding: 8px; text-align: center; vertical-align: middle;">removes any hour with more than 100 mm</td><td style="padding: 8px; text-align: center; vertical-align: middle;">a glitch in the gauge or its transmission</td><td style="padding: 8px; text-align: center; vertical-align: middle;"><code>MAX_RAIN_IN_ONE_HOUR_MM</code></td></tr>
    <tr><td style="padding: 8px; text-align: center; vertical-align: middle;">2. Blocked gauge</td><td style="padding: 8px; text-align: center; vertical-align: middle;">removes a whole month when the gauge caught less than 10% of the rain IMERG saw there</td><td style="padding: 8px; text-align: center; vertical-align: middle;">leaves or insects block the funnel: the gauge reports 0 while it rains, which looks like "dry", not "missing"</td><td style="padding: 8px; text-align: center; vertical-align: middle;"><code>BLOCKED_BELOW_PERCENT</code></td></tr>
    <tr><td style="padding: 8px; text-align: center; vertical-align: middle;">3. Clock</td><td style="padding: 8px; text-align: center; vertical-align: middle;">slides the gauge up to 3 hours either way against IMERG; it must match best where it is</td><td style="padding: 8px; text-align: center; vertical-align: middle;">a gauge clock set wrong, or in local time instead of UTC</td><td style="padding: 8px; text-align: center; vertical-align: middle;"><code>CLOCK_CHECK_HOURS</code></td></tr>
    <tr><td style="padding: 8px; text-align: center; vertical-align: middle;">4. Enough data</td><td style="padding: 8px; text-align: center; vertical-align: middle;">the gauge needs at least 2 years of data</td><td style="padding: 8px; text-align: center; vertical-align: middle;">a few months of data is too few storms to judge a product</td><td style="padding: 8px; text-align: center; vertical-align: middle;"><code>MIN_YEARS_OF_DATA</code></td></tr>
  </tbody>
</table>

Checks 1 and 2 remove bad data; a gauge that fails check 3 or 4 is not used at all. Of the gauges that pass, the one
with the most data is the **main** gauge, shown in the detailed figures.

In [ ]:
# Step 4: quality control (the gauges are checked against the first product in PRODUCTS).
qc, clean, selected, MAIN = quality_control(gauges, hourly, PRODUCTS[0], MAX_RAIN_IN_ONE_HOUR_MM, BLOCKED_BELOW_PERCENT,
                                            CLOCK_CHECK_HOURS, MIN_YEARS_OF_DATA)
qc

<div style="background:#9E1B32; border-left:6px solid #F0627A; padding:12px 16px; border-radius:8px; margin-top:0; margin-bottom:0;">
  <h3 style="margin:0; font-size:20px; font-weight:700; color:#fff5f7;">
    5. Hourly scores: how well does each product match each gauge?
  </h3>
</div>

Both products are scored on **exactly the same hours**: those where the gauge and both products have data.

<table style="border-collapse: collapse; font-size: 14px;">
  <thead>
    <tr style="background-color: #ffffff; color: #9E1B32;">
      <th style="padding: 8px; text-align: center; vertical-align: middle; color: #9E1B32; border-bottom: 2px solid #9E1B32;">Score</th>
      <th style="padding: 8px; text-align: center; vertical-align: middle; color: #9E1B32; border-bottom: 2px solid #9E1B32;">Meaning</th>
      <th style="padding: 8px; text-align: center; vertical-align: middle; color: #9E1B32; border-bottom: 2px solid #9E1B32;">Perfect</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="padding: 8px; text-align: center; vertical-align: middle;">ratio</td><td style="padding: 8px; text-align: center; vertical-align: middle;">product total / gauge total</td><td style="padding: 8px; text-align: center; vertical-align: middle;">1</td></tr>
    <tr><td style="padding: 8px; text-align: center; vertical-align: middle;">r</td><td style="padding: 8px; text-align: center; vertical-align: middle;">correlation: does the product rise and fall with the gauge?</td><td style="padding: 8px; text-align: center; vertical-align: middle;">1</td></tr>
    <tr><td style="padding: 8px; text-align: center; vertical-align: middle;">KGE</td><td style="padding: 8px; text-align: center; vertical-align: middle;">Kling–Gupta efficiency: correlation, bias and variability in one number (&gt; ~0.5 is usually called good)</td><td style="padding: 8px; text-align: center; vertical-align: middle;">1</td></tr>
    <tr><td style="padding: 8px; text-align: center; vertical-align: middle;">POD</td><td style="padding: 8px; text-align: center; vertical-align: middle;">share of the gauge's rain hours (≥ 1 mm) the product also had</td><td style="padding: 8px; text-align: center; vertical-align: middle;">1</td></tr>
    <tr><td style="padding: 8px; text-align: center; vertical-align: middle;">FAR</td><td style="padding: 8px; text-align: center; vertical-align: middle;">share of the product's rain hours the gauge did not have</td><td style="padding: 8px; text-align: center; vertical-align: middle;">0</td></tr>
    <tr><td style="padding: 8px; text-align: center; vertical-align: middle;">CSI</td><td style="padding: 8px; text-align: center; vertical-align: middle;">hits / (hits + misses + false alarms)</td><td style="padding: 8px; text-align: center; vertical-align: middle;">1</td></tr>
    <tr><td style="padding: 8px; text-align: center; vertical-align: middle;">heavy_ratio</td><td style="padding: 8px; text-align: center; vertical-align: middle;">product / gauge on heavy hours (≥ 10 mm at the gauge) only</td><td style="padding: 8px; text-align: center; vertical-align: middle;">1</td></tr>
  </tbody>
</table>

<div style="background-color:rgba(184,34,61,0.12); border-left: 5px solid #E04A63; padding: 14px; border-radius: 8px; font-family: 'Segoe UI', sans-serif; font-size: 15px; margin-top: 10px;">
  <strong>independent = False</strong> means the product may already contain that gauge, so the score flatters the product: it is partly comparing the gauge with itself.
</div>

In [ ]:
# Step 5: hourly scores at every gauge that passed QC.
score_table, hourly_at = hourly_scores(selected, clean, hourly, PRODUCTS, LABEL, NETWORKS_USED_BY, WET_HOUR_MM,
                                       HEAVY_HOUR_MM)
score_table.round(2)

<div style="background:#9E1B32; border-left:6px solid #F0627A; padding:12px 16px; border-radius:8px; margin-top:0; margin-bottom:0;">
  <h3 style="margin:0; font-size:20px; font-weight:700; color:#fff5f7;">
    6. Visual Evaluation
  </h3>
</div>

Each figure has its own cell.


<h4 style="background-color:rgba(184,34,61,0.18); padding:8px 12px; border-left: 5px solid #E04A63; border-radius:6px; font-family:'Segoe UI',sans-serif; font-size:16px; margin-top:16px;">
🔹 6.1 Hourly KGE at every gauge
</h4>

<p style="font-family:'Segoe UI',sans-serif; font-size:15px; margin-top:6px;">
Does the verdict depend on which gauge you trust? One pair of bars per gauge; hollow, hatched bars = not an independent test.
</p>

In [ ]:
# Figure 1: hourly KGE of each product at each gauge.
plot_kge_by_gauge(selected, score_table, PRODUCTS, LABEL, COLOUR, NETWORKS_USED_BY, BASIN)

<h4 style="background-color:rgba(184,34,61,0.18); padding:8px 12px; border-left: 5px solid #E04A63; border-radius:6px; font-family:'Segoe UI',sans-serif; font-size:16px; margin-top:16px;">
🔹 6.2 Cumulative rainfall at the main gauge
</h4>

<p style="font-family:'Segoe UI',sans-serif; font-size:15px; margin-top:6px;">
The running total, on hours when the gauge and both products have data. Where two lines drift apart, one of them is
biased. Dashed = not an independent test.
</p>

In [ ]:
# Figure 2: cumulative rainfall at the main gauge.
plot_cumulative_rainfall(hourly_at, MAIN, PRODUCTS, LABEL, COLOUR, NETWORKS_USED_BY, BASIN, BASIN_NAME)

<h4 style="background-color:rgba(184,34,61,0.18); padding:8px 12px; border-left: 5px solid #E04A63; border-radius:6px; font-family:'Segoe UI',sans-serif; font-size:16px; margin-top:16px;">
🔹 6.3 Hourly rain: product vs main gauge
</h4>

<p style="font-family:'Segoe UI',sans-serif; font-size:15px; margin-top:6px;">
Each dot is one hour with rain at the gauge or in the product. On the dashed 1:1 line, they agree exactly.
</p>

In [ ]:
# Figure 3: hourly scatter at the main gauge (rainy hours only).
plot_hourly_scatter(hourly_at, MAIN, score_table, PRODUCTS, LABEL, COLOUR, BASIN)

<h4 style="background-color:rgba(184,34,61,0.18); padding:8px 12px; border-left: 5px solid #E04A63; border-radius:6px; font-family:'Segoe UI',sans-serif; font-size:16px; margin-top:16px;">
🔹 6.4 One storm, hour by hour
</h4>

<p style="font-family:'Segoe UI',sans-serif; font-size:15px; margin-top:6px;">
The wettest day at the main gauge, with a day either side. Is the difference between gauge and product in the
<strong>timing</strong> or in the <strong>amount</strong>?
</p>

In [ ]:
# Figure 4: the wettest day at the main gauge, hour by hour (try your own date: add day=pd.Timestamp("2020-04-12")).
plot_wettest_storm(hourly_at, MAIN, PRODUCTS, LABEL, COLOUR, BASIN)

<h4 style="background-color:rgba(184,34,61,0.18); padding:8px 12px; border-left: 5px solid #E04A63; border-radius:6px; font-family:'Segoe UI',sans-serif; font-size:16px; margin-top:16px;">
🔹 6.5 Hourly KGE year by year at the main gauge
</h4>

<p style="font-family:'Segoe UI',sans-serif; font-size:15px; margin-top:6px;">
Over the years the radars, the gauges and the methods change. Does a product get better or worse? A year is shown
when the gauge has at least half a year of hours.
</p>

In [ ]:
# Figure 5: hourly KGE in each year at the main gauge.
yearly_kge = plot_kge_per_year(hourly_at, MAIN, PRODUCTS, LABEL, COLOUR, BASIN)
yearly_kge.round(2).T

[**LETS GO TO THE NEXT PART**](./03.model_run.ipynb)